In [ ]:
import pandas as pd
import numpy as np

In [ ]:
pip install gurobipy

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import gurobipy as gp
from gurobipy import Env, GRB

ACCESS_ID = "f1e4590e-58ce-4268-8ebf-2fc9d2d35608"
SECRET = "29915ece-fd7d-42ba-b670-26f0e7ff0612"
LICENSEID = 2726134

env = Env(empty=True)
env.setParam('WLSAccessID', ACCESS_ID)
env.setParam('WLSSecret', SECRET)
env.setParam('LicenseID', LICENSEID)
env.start()

m = gp.Model('Childcare_MinCost_Optimization', env=env)

Set parameter Username
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2726134
Academic license 2726134 - for non-commercial use only - registered to xw___@columbia.edu


In [ ]:
print("Gurobi version:", gp.gurobi.version())

Gurobi version: (11, 0, 3)


In [ ]:
from gurobipy import GRB
from pathlib import Path
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

# ==== 1. Data Loading & Sets====

In [ ]:
DATA_PATH = Path('/content/drive/MyDrive/Group 13 - Project 1')
INPUT_CSV = DATA_PATH /'Final Results'
Capacity_by_facilities = DATA_PATH / 'Final Results/child_care_regulated.csv'
Potential_locations = DATA_PATH / 'Final Results/potential_locations.csv'

Capacity_by_facilities = 'child_care_regulated.csv'
Potential_locations =  'potential_locations.csv'

In [ ]:
df_all = pd.read_csv('Childcare Deserts FINAL.csv')
df_fac = pd.read_csv(Capacity_by_facilities)
df_loc = pd.read_csv(Potential_locations)

In [ ]:
df_all

,Unnamed: 0,zipcode,Center Exists,employment rate,average income,zipcode group 1,Demand Type,total_capacity,under-5_capacity,0-4,5-9,10-12,Total Kid Count,Desert
0,0,10001,Y,0.595097,102878.033603,100,Normal,609.0,0.0,744.0,784.0,565.2,2093.2,Y
1,1,10005,Y,0.665833,121437.713311,100,High,39.0,0.0,484.0,204.0,137.4,825.4,Y
2,2,10007,Y,0.528910,138853.904282,100,Normal,284.0,0.0,605.0,451.0,174.0,1230.0,Y
3,3,10010,Y,0.492749,116272.698810,100,Normal,234.0,0.0,1422.0,1592.0,568.8,3582.8,Y
4,4,10012,Y,0.538273,111131.786340,100,Normal,24.0,0.0,613.0,161.0,244.2,1018.2,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1308,1308,14892,Y,0.565556,55246.478873,148,High,52.0,9.6,237.0,373.0,255.6,865.6,Y
1309,1309,14893,N,0.565556,59311.089615,148,High,0.0,0.0,0.0,0.0,10.8,10.8,Y
1310,1310,14894,N,0.565556,54025.423729,148,High,0.0,0.0,85.0,104.0,19.8,208.8,Y
1311,1311,14897,Y,0.565556,54044.117647,148,High,16.0,12.8,51.0,80.0,33.0,164.0,Y


In [ ]:
df_fac

,Unnamed: 0,facility_id,program_type,facility_status,facility_name,city,zipcode,school_district_name,infant_capacity,toddler_capacity,preschool_capacity,school_age_capacity,children_capacity,total_capacity,latitude,longitude,under-5_capacity
0,0,2416,FDC,Registration,"Bohrer, Barbara",Clinton,13323,Clinton,0,0,0,2,6,8,NaN,NaN,0.0
1,1,5555,FDC,Registration,"Matey, Sally",Jamestown,14701,Jamestown,0,0,0,2,6,8,NaN,NaN,0.0
2,2,9066,FDC,Registration,"Copeland, Denise",Wappingers Falls,12590,Wappingers,0,0,0,2,6,8,NaN,NaN,0.0
3,3,40163,DCC,License,"Head Start of Rockland, Inc.",Nyack,10960,Nyack,0,10,110,0,0,120,41.089425,-73.920413,76.0
4,4,41016,SACC,Registration,"School's Out, Inc.",Glenmont,12077,Bethlehem,0,0,0,75,0,75,42.607043,-73.788606,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15599,15599,892735,GFDC,License,LITTLE LILIES GROUP FAMILY DAYCARE LLC.,Bronx,10462,Bronx 11,0,0,0,4,12,16,40.854317,-73.864996,0.0
15600,15600,897263,GFDC,License,"Cummings, Darlene",Brooklyn,11205,Brooklyn 13,0,0,0,0,10,10,40.696940,-73.977121,0.0
15601,15601,901966,GFDC,License,"Pascal Genao, Angela",Yonkers,10705,Yonkers,0,0,0,4,12,16,40.910754,-73.893528,0.0
15602,15602,892455,GFDC,License,"Warnakulasuriya, Sajeeka",Staten Island,10303,Richmond 31,0,0,0,4,12,16,40.628144,-74.156228,0.0


In [ ]:
# Standardize the zipcodes and numeric columns
df_all['zipcode'] = (df_all['zipcode'].astype(str)
                  .str.extract(r'(\d+)')[0]
                  .fillna('')
                  .str.zfill(5))

for c in ['facility_capacity', 'under5_capacity', 'total_capacity', '0-4', 'Total Kid Count']:
    if c in df_all.columns:
        df_all[c] = pd.to_numeric(df_all[c], errors='coerce').fillna(0.0)

df_loc['zipcode'] = (df_loc['zipcode'].astype(str)
                  .str.extract(r'(\d+)')[0]
                  .fillna('')
                  .str.zfill(5))

In [ ]:
display(df_loc)

,zipcode,latitude,longitude
0,00501,40.816376,-73.040796
1,00501,40.817455,-73.044202
2,00501,40.813873,-73.042182
3,00501,40.814152,-73.037265
4,00501,40.824673,-73.047431
...,...,...,...
215395,14905,42.077220,-76.849045
215396,14905,42.092659,-76.838969
215397,14905,42.091170,-76.830071
215398,14905,42.077221,-76.834010


In [ ]:
# Build a facility table with unique pairs of (zipcode, facility_id)
df_fac = (
    df_fac.dropna(subset=['facility_id','latitude'])
      .assign(zipcode = df_fac['zipcode'].astype(str).str.extract(r'(\d+)')[0].fillna('').str.zfill(5),
          facility_id = df_fac['facility_id'].astype(str).str.strip()
      )
      .groupby(['zipcode', 'facility_id'], as_index=False)
      .agg(
          facility_capacity = ('total_capacity', 'max'),
          under5_capacity = ('under-5_capacity', 'max'),
          latitude = ('latitude', 'first'),
          longitude = ('longitude', 'first')
      )
)

In [ ]:
display(df_fac)

,zipcode,facility_id,facility_capacity,under5_capacity,latitude,longitude
0,10001,229433,88,0.0,40.748836,-73.999810
1,10001,292419,79,0.0,40.749247,-74.001598
2,10001,350076,8,0.0,40.748296,-74.001263
3,10001,661697,16,0.0,40.748911,-74.001546
4,10001,827488,8,0.0,40.747845,-73.989419
...,...,...,...,...,...,...
15009,14905,702559,8,0.0,42.082112,-76.855004
15010,14905,808547,16,0.0,42.076661,-76.836433
15011,14905,821932,16,0.0,42.087195,-76.838861
15012,14905,843963,52,46.0,42.099337,-76.826906


In [ ]:
missing_summary = (
    df_fac.isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_count"})
)
missing_summary

,column,missing_count
0,zipcode,0
1,facility_id,0
2,facility_capacity,0
3,under5_capacity,0
4,latitude,0
5,longitude,0


In [ ]:
# Make sure these columns are string types
df_fac['zipcode'] = df_fac['zipcode'].astype(str).str.zfill(5)
df_fac['facility_id'] = df_fac['facility_id'].astype(str).str.strip()

# Drop missing IDs
# df_fac = df_fac[df_fac['facility_id'].str.lower() != 'nan']

# Drop duplicate facility ids
before = len(df_fac)
df_fac = df_fac.drop_duplicates(subset=['zipcode','facility_id'], keep='first')
after = len(df_fac)
print(f"Removed {before - after} duplicate facility entries; remaining: {after}")

# Verify duplicates were dropped and get # of unique facility ids
dups = df_fac.duplicated(subset=['zipcode','facility_id']).sum()
if dups > 0:
    print(f"Still {dups} duplicates exist — something's wrong with your facility IDs.")
else:
    print("All (zipcode, facility_id) pairs unique.")

Removed 0 duplicate facility entries; remaining: 15014
All (zipcode, facility_id) pairs unique.


In [ ]:
missing_summary = (
    df_all.isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_count"})
)
missing_summary

,column,missing_count
0,Unnamed: 0,0
1,zipcode,0
2,Center Exists,0
3,employment rate,0
4,average income,0
5,zipcode group 1,0
6,Demand Type,0
7,total_capacity,0
8,under-5_capacity,0
9,0-4,0


In [ ]:
missing_summary = (
    df_loc.isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_count"})
)
missing_summary

,column,missing_count
0,zipcode,0
1,latitude,0
2,longitude,0


In [ ]:
# Keep only the columns we need
need_cols = ['zipcode','Demand Type','under-5_capacity','total_capacity','0-4','Total Kid Count','Center Exists']
df_all = df_all[need_cols].copy()
# Build a table in this format: (unique (zipcode, facility_id))
df_all = (df_all.assign(zipcode = df_all['zipcode'].astype(str).str.extract(r'(\d+)')[0].fillna('').str.zfill(5),))

In [ ]:
df_all

,zipcode,Demand Type,under-5_capacity,total_capacity,0-4,Total Kid Count,Center Exists
0,10001,Normal,0.0,609.0,744.0,2093.2,Y
1,10005,High,0.0,39.0,484.0,825.4,Y
2,10007,Normal,0.0,284.0,605.0,1230.0,Y
3,10010,Normal,0.0,234.0,1422.0,3582.8,Y
4,10012,Normal,0.0,24.0,613.0,1018.2,Y
...,...,...,...,...,...,...,...
1308,14892,High,9.6,52.0,237.0,865.6,Y
1309,14893,High,0.0,0.0,0.0,10.8,N
1310,14894,High,0.0,0.0,85.0,208.8,N
1311,14897,High,12.8,16.0,51.0,164.0,Y


In [ ]:
Z_all  = sorted(df_all['zipcode'].astype(str).str.zfill(5).unique().tolist()) # set of all zipcodes both existing and potential from data cleaning csv
Z_cand = sorted(df_loc['zipcode'].astype(str).str.zfill(5).unique().tolist()) # set of all zipcodes for candidate/potential locations
Z_has_fac = sorted(df_fac['zipcode'].astype(str).str.zfill(5).unique().tolist()) # set of all zipcodes for existing facilities

F = list(zip(df_fac['zipcode'].astype(str).str.zfill(5), df_fac['facility_id'].astype(str).str.strip())) # set of (zipcode, facility_id) for existing facilities

# Map each zipcode to list of existing facilities that are located in that zipcode
from collections import defaultdict
zip_to_fac = defaultdict(list)
for z,f in F:
    zip_to_fac[z].append((z,f))


# ==== 2. Parameter Calculation ====

In [ ]:
# Existing capacity for each facility
nF_tot = {(r.zipcode, str(r.facility_id)): float(r.facility_capacity) for r in df_fac[['zipcode','facility_id','facility_capacity']].itertuples(index=False)}
nF_u5 = {(r.zipcode, str(r.facility_id)): float(r.under5_capacity) for r in df_fac[['zipcode','facility_id','under5_capacity']].itertuples(index=False)}

In [ ]:
# Current total capacity for each zipcode across all facilities in that zipcode
zip_tot_exists = df_fac.groupby('zipcode')['facility_capacity'].sum().to_dict()
zip_u5_exists = df_fac.groupby('zipcode')['under5_capacity'].sum().to_dict()

In [ ]:
# Demand thresholds - High = 1/2, Normal = 1/3
def total_share(x):
    return 0.50 if str(x).strip().lower()=='high' else (1.0/3.0)

In [ ]:
# Under-5 needs 2/3 * (0–4 population)
needTot = {}
needU5  = {}
zip_info = df_all.groupby('zipcode', as_index=False)[['Demand Type','Total Kid Count','0-4']].first()
for r in zip_info.itertuples(index=False):
    z = r[0]
    dem_type  = r[1]
    kid_total = float(r[2])
    pop_0_4 = float(r[3])
    needTot[z] = int(np.ceil(total_share(dem_type) * kid_total))
    needU5[z] = int(np.ceil((2.0/3.0) * pop_0_4))

# ==== 3. Model Construction ====

In [ ]:
m = gp.Model('Childcare_MinCost_FacilityExpand_ZipBuild',env=env)

In [ ]:
# Decision variables: Building new facilites by zipode
B_s = m.addVars(Z_cand, vtype=GRB.INTEGER, name='Build_Small') # +100 total, +50 U5
B_m = m.addVars(Z_cand, vtype=GRB.INTEGER, name='Build_Medium') # +200 total, +100 U5
B_l = m.addVars(Z_cand, vtype=GRB.INTEGER, name='Build_Large') # +400 total, +200 U5
U5_new = m.addVars(Z_cand, vtype=GRB.INTEGER, name='U5_From_New') # U5 share from new builds

In [ ]:
# Decision variables: Facility expansion by zipcode and facility_id
E_tot = m.addVars(F, vtype=GRB.INTEGER, name='Expand_Total') # per-facility total expansion
E_u5 = m.addVars(F, vtype=GRB.INTEGER, name='Expand_U5') # per-facility U5 expansion

In [ ]:
# Expansion capacity constraints for existing facilities
for (z,f) in F:
    n = nF_tot[(z,f)]
    m.addConstr(E_u5[(z,f)] <= E_tot[(z,f)], name=f'U5_le_Total[{z},{f}]')
    if n <= 0:
        m.addConstr(E_tot[(z,f)] == 0.0, name=f'NoExpandTot[{z},{f}]')
        m.addConstr(E_u5[(z,f)] == 0.0, name=f'NoExpandU5[{z},{f}]')
    else:
        m.addConstr(E_tot[(z,f)] <= min(1.2*n, 500.0), name=f'CapTot[{z},{f}]')
        m.addConstr(E_u5[(z,f)] <= E_tot[(z,f)], name=f'CapU5[{z},{f}]')

In [ ]:
# Coverage constraints per zip code: existing + expansion + new >= need
for z in Z_all:
    tot_exist = gp.quicksum(nF_tot[(zz,ff)] for (zz,ff) in zip_to_fac.get(z, []))
    u5_exist = gp.quicksum(nF_u5[(zz,ff)] for (zz,ff) in zip_to_fac.get(z, []))

    tot_exp = gp.quicksum(E_tot[(zz,ff)] for (zz,ff) in zip_to_fac.get(z, []))
    u5_exp = gp.quicksum(E_u5[(zz,ff)] for (zz,ff) in zip_to_fac.get(z, []))

    # only if z is part of Z_build otherwise set equal to 0
    cap_new = (100*B_s[z] + 200*B_m[z] + 400*B_l[z]) if z in Z_cand else gp.LinExpr(0.0)
    u5_from_new = (U5_new[z]) if z in Z_cand else gp.LinExpr(0.0)

    m.addConstr(tot_exist + tot_exp + cap_new >= needTot[z], name=f"TotalCover[{z}]")
    m.addConstr(u5_exist + u5_exp + u5_from_new >= needU5[z], name=f"U5Cover[{z}]")

    # Under 5 allocation capacity from new builds per zipcode
    if z in Z_cand:
      m.addConstr(U5_new[z] <=  50*B_s[z] + 100*B_m[z] + 200*B_l[z], name=f'U5CapFromBuild[{z}]')

In [ ]:
# Objective Function
build_cost = gp.quicksum(65000*B_s[z] + 95000*B_m[z] + 115000*B_l[z] for z in Z_cand)
expand_base_cost = gp.quicksum(((20000.0 + 200.0*nF_tot[(z,f)]) * (E_tot[(z,f)]/nF_tot[(z,f)])) if nF_tot[(z,f)] > 0 else 0.0 for (z,f) in F)
u5_equip_cost = gp.quicksum(100.0*U5_new[z] for z in Z_cand) + gp.quicksum(100.0*E_u5[(z,f)] for (z,f) in F)

m.setObjective(build_cost + expand_base_cost + u5_equip_cost, GRB.MINIMIZE)

# ==== 4. Optimization ====

In [ ]:
m.Params.OutputFlag = 1
m.optimize()

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[x86] - Darwin 22.3.0 22D68)

CPU model: Intel(R) Core(TM) i7-1068NG7 CPU @ 2.30GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Academic license 2726134 - for non-commercial use only - registered to xw___@columbia.edu
Optimize a model with 48981 rows, 38644 columns and 97153 nonzeros
Model fingerprint: 0x2d5e0354
Variable types: 0 continuous, 38644 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+02]
  Objective range  [1e+02, 1e+05]
  Bounds range     [0e+00, 0e+00]
  RHS range        [4e-01, 7e+03]
Found heuristic solution: objective 3.142503e+08
Presolve removed 48729 rows and 38142 columns
Presolve time: 0.71s
Presolved: 252 rows, 502 columns, 1004 nonzeros
Found heuristic solution: objective 2.233003e+08
Variable types: 0 continuous, 502 integer (0 binary)

Root relaxation: objective 2.221688e+08, 258 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current 

# ==== 5. Collect Results  ====

In [ ]:
  # Facility level expansions
fac_rows = []
for (z,f) in F:
    fac_rows.append({
        'zipcode': z,
        'facility_id': f,
        'E_total': float(E_tot[(z,f)].X),
        'E_u5': float(E_u5[(z,f)].X),
    })
df_fac_sol = pd.DataFrame(fac_rows)

In [ ]:
# adding results we want in the csv file
zip_rows = []
for z in Z_all:
    cost_build = (65000*B_s[z].X + 95000*B_m[z].X + 115000*B_l[z].X) if z in Z_cand else 0

    cost_expand_base = 0.0
    for (zz,ff) in zip_to_fac.get(z, []):
        if nF_tot[(zz,ff)] > 0:
            cost_expand_base += (20000.0 + 200.0*nF_tot[(zz,ff)]) * (E_tot[(zz,ff)].X/nF_tot[(zz,ff)])

    cost_u5_equip = 100.0*U5_new[z].X if z in Z_cand else 0
    # for (zz, ff) in zip_to_fac.get(z, []):
    #     cost_u5_equip += 100.0*E_u5[(zz, ff)].X
    cost_u5_equip += 100.0 * sum(E_u5[(zz,ff)].X for (zz,ff) in F if zz == z)

    zip_rows.append({
        'zipcode': z,
        'Build_Small': int(round(B_s[z].X)) if z in Z_cand else 0,
        'Build_Medium': int(round(B_m[z].X)) if z in Z_cand else 0,
        'Build_Large': int(round(B_l[z].X)) if z in Z_cand else 0,
        'U5_From_New': float(U5_new[z].X) if z in Z_cand else 0,
        'Cost_Build': float(cost_build),
        'Cost_ExpandBase': float(cost_expand_base),
        'Cost_U5_Equip': float(cost_u5_equip),
        'Total_Cost': float(cost_build + cost_expand_base + cost_u5_equip),
    })
df_zip_sol = pd.DataFrame(zip_rows)

In [ ]:
def sum_by_zip(d):
    # Initialize with all the zipcodes that we want in the report
    acc = {z: 0.0 for z in Z_all}

    # Support dicts keyed by (z,f) or by z
    for k, val in d.items():
        if isinstance(k, tuple): # (z,f)
            z = k[0]
        else: # z
            z = k
        if z in acc:
            acc[z] += float(val)
    return acc

# Build the per zipcode aggregates
sum_exist_tot = sum_by_zip(nF_tot)
sum_exist_u5 = sum_by_zip(nF_u5)
sum_E_tot = sum_by_zip({k: E_tot[k].X for k in F})
sum_E_u5 = sum_by_zip({k: E_u5[k].X for k in F})

# makes sure df_zip_sol lists all zipcodes
if set(df_zip_sol['zipcode']) != set(Z_all):
    df_zip_sol = (pd.DataFrame({'zipcode': sorted(Z_all)}).merge(df_zip_sol, on='zipcode', how='left').fillna(0))

df_zip_sol['Existing_Total'] = df_zip_sol['zipcode'].map(sum_exist_tot)
df_zip_sol['Existing_U5'] = df_zip_sol['zipcode'].map(sum_exist_u5)
df_zip_sol['Expand_Total'] = df_zip_sol['zipcode'].map(sum_E_tot)
df_zip_sol['Expand_U5'] = df_zip_sol['zipcode'].map(sum_E_u5)

df_zip_sol['New_Total'] = (100*df_zip_sol['Build_Small'] + 200*df_zip_sol['Build_Medium'] + 400*df_zip_sol['Build_Large'])
df_zip_sol['New_U5'] = df_zip_sol['U5_From_New']

df_zip_sol['Need_Total'] = df_zip_sol['zipcode'].map(needTot)
df_zip_sol['Need_U5'] = df_zip_sol['zipcode'].map(needU5)

# Check if we met demand and if desert has been eliminated for each zipcode
df_zip_sol['Meet_Total_Demand'] = (df_zip_sol['Existing_Total'] + df_zip_sol['Expand_Total'] + df_zip_sol['New_Total']) >= df_zip_sol['Need_Total']

df_zip_sol['Meet_U5_Demand'] = (df_zip_sol['Existing_U5'] + df_zip_sol['Expand_U5'] + df_zip_sol['New_U5']) >= df_zip_sol['Need_U5']

df_zip_sol['Desert_Eliminated'] = (df_zip_sol['Meet_Total_Demand'] & df_zip_sol['Meet_U5_Demand'])


In [ ]:
print("\nPreview ZIP-level:")
display(df_zip_sol.head(10))
print("Preview Facility-level:")
display(df_fac_sol.head(10))


Preview ZIP-level:


,zipcode,Build_Small,Build_Medium,Build_Large,U5_From_New,Cost_Build,Cost_ExpandBase,Cost_U5_Equip,Total_Cost,Existing_Total,Existing_U5,Expand_Total,Expand_U5,New_Total,New_U5,Need_Total,Need_U5,Meet_Total_Demand,Meet_U5_Demand,Desert_Eliminated
0,10001,0,0,0,0.0,0.0,156798.143865,49600.0,206398.143865,609.0,0.0,496.0,496.0,0,0.0,698,496,True,True,True
1,10005,0,1,1,300.0,210000.0,16394.871795,32300.0,258694.871795,39.0,0.0,23.0,23.0,600,300.0,413,323,True,True,True
2,10007,0,0,1,200.0,115000.0,63466.666667,40400.0,218866.666667,284.0,0.0,204.0,204.0,400,200.0,410,404,True,True,True
3,10010,0,0,4,800.0,460000.0,50742.857143,94800.0,605542.857143,234.0,0.0,148.0,148.0,1600,800.0,1195,948,True,True,True
4,10012,0,0,2,400.0,230000.0,13050.000000,40900.0,283950.000000,24.0,0.0,9.0,9.0,800,400.0,340,409,True,True,True
5,10013,0,0,2,400.0,230000.0,131911.413756,83300.0,445211.413756,435.0,0.0,433.0,433.0,800,400.0,1061,833,True,True,True
6,10016,0,0,2,400.0,230000.0,198920.795107,119900.0,548820.795107,823.0,0.0,799.0,799.0,800,400.0,1302,1199,True,True,True
7,10017,1,0,0,50.0,65000.0,42778.723404,12900.0,120678.723404,107.0,0.0,79.0,79.0,100,50.0,255,129,True,True,True
8,10018,0,0,1,162.0,115000.0,0.000000,16200.0,131200.000000,0.0,0.0,0.0,0.0,400,162.0,268,162,True,True,True
9,10019,0,0,1,200.0,115000.0,175867.138878,73300.0,364167.138878,591.0,0.0,533.0,533.0,400,200.0,1047,733,True,True,True


Preview Facility-level:


,zipcode,facility_id,E_total,E_u5
0,10001,229433,105.0,105.0
1,10001,292419,0.0,0.0
2,10001,350076,0.0,0.0
3,10001,661697,0.0,0.0
4,10001,827488,0.0,0.0
5,10001,837329,0.0,0.0
6,10001,837597,41.0,41.0
7,10001,893683,350.0,350.0
8,10001,912862,0.0,0.0
9,10002,101512,0.0,0.0


In [ ]:
if m.status == GRB.OPTIMAL:
    print(f"Optimization complete.")
    print(f"Total minimum cost = ${m.objVal:,.2f}")
else:
    print(f"Model status: {m.status} — no optimal solution found.")

Optimization complete.
Total minimum cost = $222,168,837.22


In [ ]:
import os
from pathlib import Path

In [ ]:
SAVE_DIR = Path('/content/drive/MyDrive/Group 13 - Project 1/Data Cleaning')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
preferred_cols = ['zipcode','Build_Small','Build_Medium','Build_Large',
                  'New_Total','U5_From_New','New_U5','Existing_Total','Expand_Total',
                  'Need_Total','Meet_Total_Demand','Existing_U5','Expand_U5','Need_U5','Meet_U5_Demand',
                  'Desert_Eliminated','Cost_Build','Cost_ExpandBase','Cost_U5_Equip','Total_Cost']

In [ ]:
cols_to_export = [c for c in preferred_cols if c in df_zip_sol.columns]
zip_out = df_zip_sol.loc[:, cols_to_export].copy()

In [ ]:
out_path = SAVE_DIR / 'optimization_result_problem1_final.csv'
zip_out.to_csv(out_path, index=False)

print(f"ZIP-level results saved to: {out_path}")
print("Preview:")
display(zip_out.head(20))

ZIP-level results saved to: optimization_result_problem1_final.csv
Preview:


,zipcode,Build_Small,Build_Medium,Build_Large,New_Total,U5_From_New,New_U5,Existing_Total,Expand_Total,Need_Total,Meet_Total_Demand,Existing_U5,Expand_U5,Need_U5,Meet_U5_Demand,Desert_Eliminated,Cost_Build,Cost_ExpandBase,Cost_U5_Equip,Total_Cost
0,10001,0,0,0,0,0.0,0.0,609.0,496.0,698,True,0.0,496.0,496,True,True,0.0,156798.143865,49600.0,2.063981e+05
1,10005,0,1,1,600,300.0,300.0,39.0,23.0,413,True,0.0,23.0,323,True,True,210000.0,16394.871795,32300.0,2.586949e+05
2,10007,0,0,1,400,200.0,200.0,284.0,204.0,410,True,0.0,204.0,404,True,True,115000.0,63466.666667,40400.0,2.188667e+05
3,10010,0,0,4,1600,800.0,800.0,234.0,148.0,1195,True,0.0,148.0,948,True,True,460000.0,50742.857143,94800.0,6.055429e+05
4,10012,0,0,2,800,400.0,400.0,24.0,9.0,340,True,0.0,9.0,409,True,True,230000.0,13050.000000,40900.0,2.839500e+05
5,10013,0,0,2,800,400.0,400.0,435.0,433.0,1061,True,0.0,433.0,833,True,True,230000.0,131911.413756,83300.0,4.452114e+05
6,10016,0,0,2,800,400.0,400.0,823.0,799.0,1302,True,0.0,799.0,1199,True,True,230000.0,198920.795107,119900.0,5.488208e+05
7,10017,1,0,0,100,50.0,50.0,107.0,79.0,255,True,0.0,79.0,129,True,True,65000.0,42778.723404,12900.0,1.206787e+05
8,10018,0,0,1,400,162.0,162.0,0.0,0.0,268,True,0.0,0.0,162,True,True,115000.0,0.000000,16200.0,1.312000e+05
9,10019,0,0,1,400,200.0,200.0,591.0,533.0,1047,True,0.0,533.0,733,True,True,115000.0,175867.138878,73300.0,3.641671e+05
